In [2]:
from pyspark.sql import functions as F, Window

# Customers - customer_id is unique
df_customers = spark.read.table("lh_bronze_olist.dbo.olist_customers_dataset")

df_customers_clean = (
    df_customers
    .dropDuplicates(["customer_id"])
)

df_customers_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("lh_silver_olist.dbo.customers")

print(f"customers: {df_customers_clean.count()} records")

StatementMeta(, 69f45b4c-1413-43e7-8aa5-6b7b90398d48, 4, Finished, Available, Finished, False)

customers: 99441 records


In [1]:
# Geolocation - deduplicate to one row per zip_code_prefix.
#
# The raw dataset has multiple rows per zip_code_prefix (different addresses
# sharing the same prefix), so a straightforward load produces duplicates in
# the dimension. Naive deduplication (e.g. distinct on prefix+city+state)
# still leaves duplicates behind, because the same location is written
# inconsistently across rows — different casing, accented vs. unaccented
# characters, and in some cases genuinely different towns sharing a prefix.
#
# This is fixed in three layered steps below:
#   1. Normalize text (case, whitespace, accents) so identical locations
#      that were written inconsistently collapse into the same group.
#   2. Aggregate per (prefix, city, state): average the coordinates across
#      all rows in the group (rather than arbitrarily picking one row),
#      and keep a row count to use as a confidence weight.
#   3. Where duplicates still remain per prefix (genuine typos, or two
#      distinct towns sharing a zip prefix), break the tie by keeping the
#      (city, state) grouping with the most supporting raw rows.


df_geo = spark.read.table("lh_bronze_olist.dbo.olist_geolocation_dataset")

accented = "áàâãäéèêëíìîïóòôõöúùûüçñÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇÑ"
plain    = "aaaaaeeeeiiiiooooouuuucnAAAAAEEEEIIIIOOOOOUUUUCN"

# Full state-code -> state-name lookup, used to enrich DimLocation later.
state_mapping = {
    "AC": "Acre", "AL": "Alagoas", "AP": "Amapa", "AM": "Amazonas",
    "BA": "Bahia", "CE": "Ceara", "DF": "Distrito Federal", "ES": "Espirito Santo",
    "GO": "Goias", "MA": "Maranhao", "MT": "Mato Grosso", "MS": "Mato Grosso do Sul",
    "MG": "Minas Gerais", "PA": "Para", "PB": "Paraiba", "PR": "Parana",
    "PE": "Pernambuco", "PI": "Piaui", "RJ": "Rio de Janeiro", "RN": "Rio Grande do Norte",
    "RS": "Rio Grande do Sul", "RO": "Rondonia", "RR": "Roraima", "SC": "Santa Catarina",
    "SP": "Sao Paulo", "SE": "Sergipe", "TO": "Tocantins"
}
state_mapping_expr = F.create_map([F.lit(x) for pair in state_mapping.items() for x in pair])

# Step 1: Normalize whitespace, casing, and accented characters.
# Two causes of "duplicate" locations live here: inconsistent casing/whitespace,
# and accented vs. unaccented spellings of the same city name (e.g. "ibiaça"
# vs "ibiaca"). Without this step, those are treated as different cities.
df_geo_normalized = (
    df_geo
    .withColumn("geolocation_city", F.trim(F.lower(F.regexp_replace(F.translate(F.col("geolocation_city"), accented, plain),"-"," "))))
    .withColumn("geolocation_state", F.trim(F.upper(F.col("geolocation_state"))))
)

# Step 2: Aggregate per (prefix, city, state).
# Average lat/lng across all rows in the group rather than picking one
# arbitrarily, since multiple raw addresses can share a prefix. row_count
# is carried forward as a confidence weight for the tie-breaker in step 3.
df_geo_grouped = (
    df_geo_normalized
    .groupBy("geolocation_zip_code_prefix", "geolocation_city", "geolocation_state")
    .agg(
        F.avg("geolocation_lat").alias("avg_lat"),
        F.avg("geolocation_lng").alias("avg_lng"),
        F.count("*").alias("row_count")
    )
)

# Step 3: Tie-breaker for prefixes that still have more than one (city, state)
# group after step 2 — either genuine typos in the source data or two distinct
# towns that legitimately share a zip prefix. Keep the group backed by the
# most raw rows (highest row_count) as the representative location.
window_spec = Window.partitionBy("geolocation_zip_code_prefix").orderBy(F.desc("row_count"))

df_geo_clean = (
    df_geo_grouped
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
    .drop("row_count", "rn")
    .withColumn("geolocation_state_name", state_mapping_expr[F.col("geolocation_state")])
)

df_geo_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("lh_silver_olist.dbo.geolocation")

print(f"geolocation: {df_geo_clean.count()} records")

StatementMeta(, d491eb73-50ae-467d-afea-0b34f28e5906, 3, Finished, Available, Finished, False)

geolocation: 19015 Zeilen


In [4]:
# Sellers - seller_id ist unique
df_sellers = spark.read.table("lh_bronze_olist.dbo.olist_sellers_dataset")

df_sellers_clean = (
    df_sellers
    .dropDuplicates(["seller_id"])
)

df_sellers_clean.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("lh_silver_olist.dbo.sellers")

print(f"sellers: {df_sellers_clean.count()} records")

StatementMeta(, 69f45b4c-1413-43e7-8aa5-6b7b90398d48, 6, Finished, Available, Finished, False)

sellers: 3095 records
